In [1]:
import sys
import getopt
import shutil
import h5py
import re
import os
import numpy as np
import pandas as pd
import scanpy as sc
import celltypist
import gc
import anndata
from celltypist import models
import h5py
import scipy.sparse as scs
from multiprocessing import Pool
import scrublet

In [2]:
h5_list = ['sc5p_donor_match']

In [3]:
in_dir = '../../reflex_data/EXP1936_5prime_gex/'

In [4]:
for h5_in in h5_list:
    print(h5_in)

    h5_path = in_dir + '/outs/filtered_feature_bc_matrix.h5'
    

    # Perform Scrublet & celltypist
    adata = sc.read_10x_h5(h5_path)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    predictions_l1 = celltypist.annotate(adata, model='/home/workspace/celltypist_dev/models/model_AIFI_PBMC_L1_ovr.pkl')
    predictions_l2 = celltypist.annotate(adata, model='/home/workspace/celltypist_dev/models/model_AIFI_PBMC_L2_ovr.pkl')
    predictions_l3 = celltypist.annotate(adata, model='/home/workspace/celltypist_dev/models/model_AIFI_PBMC_L3_multinomial.pkl')

    df = pd.DataFrame(predictions_l1.predicted_labels)
    df.columns = ['L1']
    df2 = pd.DataFrame(predictions_l2.predicted_labels)
    df2.columns = ['L2']
    df3 = pd.DataFrame(predictions_l3.predicted_labels)
    df3.columns = ['L3']

    df_merge = df.merge(df2, left_index=True, right_index=True)
    df_merge2 = df_merge.merge(df3, left_index=True, right_index=True)
    df_merge2.to_csv(path_or_buf = '5prime_celltypist_scrublet/' + h5_in+'_celltypes.csv')
    
    
    adata.shape
    scrub = scrublet.Scrublet(adata.X)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(verbose = False)

    df = pd.DataFrame(doublet_scores)
    df.columns = ['Doublet_Score']
    df2 = pd.DataFrame(predicted_doublets)
    df2.columns = ['Predicted_Doublet']

    df_merge = df.merge(df2, left_index=True, right_index=True)
    df_merge.to_csv(path_or_buf = '5prime_celltypist_scrublet/' + h5_in+'_doublets.csv')

sc5p_donor_match


🔬 Input data has 17640 cells and 38606 genes
🔗 Matching reference genes in the model
🧬 1035 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
🔬 Input data has 17640 cells and 38606 genes
🔗 Matching reference genes in the model
🧬 1824 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
🔬 Input data has 17640 cells and 38606 genes
🔗 Matching reference genes in the model
🧬 2402 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
